In [1]:
!pip install kagglehub pandas numpy scikit-learn matplotlib seaborn


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download("nikhil1e9/loan-default")

print(path)
print(os.listdir(path))

c:\Users\gurra\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\gurra\.cache\kagglehub\datasets\nikhil1e9\loan-default\versions\2
['Loan_default.csv']


In [3]:
file_path = os.path.join(path, "loan_default.csv")  # adjust if name differs
df = pd.read_csv(file_path)

df.head()

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


In [4]:
# Target
y = df['Default']

# Features
X = df.drop('Default', axis=1)

In [5]:
X = df.drop(['Default', 'LoanID'], axis=1)
y = df['Default']

In [6]:
print(X.select_dtypes(include='object').columns)

Index(['Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage',
       'HasDependents', 'LoanPurpose', 'HasCoSigner'],
      dtype='object')


In [7]:
X = pd.get_dummies(X, drop_first=True)

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [11]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200, random_state=42)

In [12]:
from sklearn.metrics import accuracy_score

print("Logistic Regression:", accuracy_score(y_test, log_model.predict(X_test)))
print("Random Forest:", accuracy_score(y_test, rf_model.predict(X_test)))

Logistic Regression: 0.885275112590562
Random Forest: 0.8853925983943607


In [13]:
from sklearn.decomposition import PCA

pca = PCA(n_components=5)
X_pca = pca.fit_transform(X)
print(X_pca.shape)

(255347, 5)


In [14]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42)
df['Cluster'] = kmeans.fit_predict(X)

In [15]:
import pickle

pickle.dump(rf_model, open("loan_rf.pkl", "wb"))
pickle.dump(log_model, open("loan_logistic.pkl", "wb"))
pickle.dump(scaler, open("loan_scaler.pkl", "wb"))
pickle.dump(X.columns, open("loan_columns.pkl", "wb"))

In [16]:
import numpy as np
import pandas as pd
import pickle

model = pickle.load(open("loan_rf.pkl", "rb"))
scaler = pickle.load(open("loan_scaler.pkl", "rb"))
columns = pickle.load(open("loan_columns.pkl", "rb"))

def predict_loan(data_dict):

    df_input = pd.DataFrame([data_dict])
    df_input = df_input.reindex(columns=columns, fill_value=0)

    df_scaled = scaler.transform(df_input)

    prob = model.predict_proba(df_scaled)[0][1] * 100

    return {
        "default_probability": round(prob, 2),
        "risk_level": "HIGH" if prob > 60 else "LOW"
    }

In [17]:
import json
import pandas as pd
import pickle

model = pickle.load(open("loan_rf.pkl", "rb"))
scaler = pickle.load(open("loan_scaler.pkl", "rb"))
columns = pickle.load(open("loan_columns.pkl", "rb"))

def lambda_handler(event, context):

    df = pd.DataFrame([event])
    df = df.reindex(columns=columns, fill_value=0)

    scaled = scaler.transform(df)

    prob = model.predict_proba(scaled)[0][1] * 100

    return {
        "statusCode": 200,
        "body": json.dumps({
            "default_probability": round(prob, 2),
            "risk_level": "HIGH" if prob > 60 else "LOW"
        })
    }

In [18]:
from sklearn.ensemble import RandomForestClassifier
import pickle

rf_model = RandomForestClassifier(
    n_estimators=50,   # reduced from 200
    max_depth=10,      # limits tree size
    random_state=42
)

rf_model.fit(X_train, y_train)

pickle.dump(rf_model, open("loan_rf.pkl", "wb"))

In [19]:
import os
print(os.path.getsize("loan_rf.pkl") / (1024 * 1024), "MB")

6.679732322692871 MB


In [20]:
import os
print("loan_rf.pkl:", os.path.getsize("loan_rf.pkl")/(1024*1024), "MB")
print("loan_scaler.pkl:", os.path.getsize("loan_scaler.pkl")/(1024*1024), "MB")
print("loan_columns.pkl:", os.path.getsize("loan_columns.pkl")/(1024*1024), "MB")

loan_rf.pkl: 6.679732322692871 MB
loan_scaler.pkl: 0.0014896392822265625 MB
loan_columns.pkl: 0.0006437301635742188 MB


In [21]:
import os

for f in [
    "loan_rf.pkl",
    "loan_scaler.pkl",
    "loan_columns.pkl"
]:
    if os.path.exists(f):
        os.remove(f)

In [22]:
import pickle

pickle.dump(rf_model, open("loan_rf.pkl", "wb"))
pickle.dump(scaler, open("loan_scaler.pkl", "wb"))
pickle.dump(X.columns, open("loan_columns.pkl", "wb"))